In [1]:
import pandas as pd
import numpy as np

# pip install category-encoders
import category_encoders as ce

In [2]:
df = pd.read_csv("../../Datasets/adult.csv")   
df = df.drop_duplicates()
df = df.replace("?", np.nan).dropna()

In [3]:
# We have our target variable in text format, so we need to convert it to binary values
y = (df["income"] == ">50K").astype(int)
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K


In [4]:
# we have 41 unique values in the native.country column
df['native.country'].nunique()

41

In [5]:
df['native.country'].value_counts()

native.country
United-States                 27487
Mexico                          606
Philippines                     188
Germany                         128
Puerto-Rico                     109
Canada                          107
El-Salvador                     100
India                           100
Cuba                             92
England                          86
Jamaica                          80
South                            71
Italy                            68
China                            68
Dominican-Republic               67
Vietnam                          64
Guatemala                        61
Japan                            59
Poland                           56
Columbia                         56
Taiwan                           42
Haiti                            42
Iran                             42
Portugal                         34
Nicaragua                        33
Peru                             30
Greece                           29
Ecuador      

### Encoder 1: TargetEncoder

In [6]:
# create a target encoder by using category encoders
encoder = ce.TargetEncoder(cols=["native.country"])

# apply the encoder, Description is going to be encoded, Sales is our target variable
df_encoded_target = encoder.fit_transform(df['native.country'], y)

# insert the encoded description averages back to the original data
df['native.country_encoded'] = df_encoded_target

# the usual process is this => you save and use the original product name
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the sales => drop the original product name
# df.drop("Description", axis=1)
df.head(5)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income,native.country_encoded
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K,0.254411
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K,0.254411
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K,0.254411
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K,0.254411
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K,0.254411


In [7]:
# Let's check the encoded values for US
df.loc[df["native.country"] == "United-States", "native.country_encoded"].iloc[0]

np.float64(0.25441117619238185)

In [8]:
# Now check the average of the target variable for US
# So the aaverage and the encoded values are same
y[df["native.country"] == "United-States"].mean()

np.float64(0.25441117619238185)

In [9]:
# Let's check the encoded values for Germany
df.loc[df["native.country"] == "Germany", "native.country_encoded"].iloc[0]

np.float64(0.3437480681266021)

In [10]:
# Same for Germany, the average and the encoded values are same
y[df["native.country"] == "Germany"].mean()

np.float64(0.34375)

### Encoder 2: CountEncoder

In [11]:
df['native.country'].value_counts().head(5)

native.country
United-States    27487
Mexico             606
Philippines        188
Germany            128
Puerto-Rico        109
Name: count, dtype: int64

In [12]:
# create a target encoder by using category encoders
encoder = ce.CountEncoder(cols=["native.country"])

# apply the encoder, Description is going to be encoded, Sales is our target variable
df_encoded_target = encoder.fit_transform(df['native.country'])

# insert the encoded description averages back to the original data
df['native.country_count_encoded'] = df_encoded_target

# the usual process is this => you save and use the original product name
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the sales => drop the original product name
# df.drop("Description", axis=1)
df[["native.country", "native.country_count_encoded"]].head(5)

# REMEMBER: if you drop the original description and after that remove all duplicates
# you will most likely have less rows in the data because of that

# the frequency/count approach is probably most effective when
# the popularity of some certain item has a meaning in ML model
# for ales data, popular items probably have more optimistic sales estimations
# (etc. depends on context/situation however)

,native.country,native.country_count_encoded
1,United-States,27487
3,United-States,27487
4,United-States,27487
5,United-States,27487
6,United-States,27487


In [13]:
df["native.country"].nunique()

41

In [14]:
df.shape

(30139, 17)

In [ ]:
# Let's check the correlation between the encoded values and the target variable
df[["native.country_encoded", "native.country_count_encoded"]].corrwith(y)

native.country_encoded          0.101229
native.country_count_encoded    0.039534
dtype: float64

### Conclusion

* I have now one numeric column per encoder instead of having 41 one-hot columns.

* Target encoding has correlation of 0.10 with income while count encoding 0.04, so target encoding is bit more informative, but neither is strong. 

* A limit of target encoding is that it uses the target, so it should be fitted on training data only.